# AGCN Presenter v0.2 — QLoRA e avaliação controlada

500 Gold (458 treino / 42 validação). Somente branch research/agcn-presenter-training-v1. Os 25 casos congelados não entram no treino. Pesos e checkpoints somente no Google Drive.

Selecione T4 GPU. Este notebook não compra recursos, não troca de conta e não altera a main. Se o Colab negar GPU por limite de uso, a execução deve parar até a GPU estar disponível.

In [ ]:
import torch
assert torch.cuda.is_available(), 'GPU indisponível: treino não iniciado'
print(torch.cuda.get_device_name(0))


## 1. Branch de pesquisa
Clone público sem token. O assert impede executar na main; o pull é somente fast-forward da branch de pesquisa.

In [ ]:
import os, pathlib, subprocess
repo = pathlib.Path('/content/AGCN-Live-Voice')
branch = 'research/agcn-presenter-training-v1'
if not repo.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--single-branch', '--branch', branch, 'https://github.com/adrse/AGCN-Live-Voice.git', str(repo)], check=True)
os.chdir(repo)
assert subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip() == branch
subprocess.run(['git', 'pull', '--ff-only', 'origin', branch], check=True)
print('Branch:', branch, 'Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## 2. Dependências e corpus
Transformers 4.57.6; FP16 na T4; BF16 somente nativo. Validação de 500 Gold, splits, hashes e exclusão dos testes antes de gastar tempo de treino.

In [ ]:
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'training/requirements-lora.txt'], check=True)
subprocess.run([sys.executable, 'training/scripts/run_presenter_v0_2_experiment.py', '--preflight-only'], check=True)


## 3. Google Drive
Use a conta que contém AGCN/Presenter/agcn-presenter-v0.1-lora. A autorização de conta acontece na interface do Google. O hash do adapter v0.1 é conferido antes do treino.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 4. Treino e três avaliações
4 épocas, learning rate 2e-4, LoRA r=16/alpha=32, acumulação 8, batch 1. Começa no Qwen original, não no adapter v0.1. Melhor checkpoint escolhido pela menor validation loss, sem usar os testes.

A execução registra logs no Drive, retoma checkpoints e executa 25 casos para Qwen original, AGCN v0.1 e AGCN v0.2 com o mesmo protocolo. Uma avaliação incompleta é reexecutada; avaliações completas são preservadas. A revisão semântica permanece pendente até analisar as 75 saídas.

Esta célula aguarda o processo terminar. Logs e checkpoints são salvos durante a execução; uma desconexão do Colab não é tratada como sucesso.

In [ ]:
subprocess.run([sys.executable, '-u', 'training/scripts/run_presenter_v0_2_experiment.py'], check=True)


## 5. Conferir resultados
O status deve indicar treino e inferência completos; revisão semântica pendente. Abra comparison_case_by_case.md para revisar fatos inventados, escassez, intenção de compra, objeção, IGNORAR, CTA, naturalidade e repetição. Métricas lexicais não substituem essa revisão.

In [ ]:
import json
results = pathlib.Path('/content/drive/MyDrive/AGCN/Presenter/Results-v0.2')
print((results / 'status.json').read_text())
print((pathlib.Path('/content/drive/MyDrive/AGCN/Presenter/agcn-presenter-v0.2-lora') / 'agcn_training_summary.json').read_text())
for name in ['qwen_original', 'agcn_v0.1', 'agcn_v0.2']:
    print(name, (results / (name + '_summary.json')).read_text())
print('Relatório:', results / 'comparison_case_by_case.md')
